In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import Delaunay
import plotly.io as pio

pio.renderers.default = "browser"


a = np.sqrt(2 * 15)
b = np.sqrt(3 * 15)
c = np.sqrt(4 * 15)

def ellipsoid_xyz(theta, phi):
    X = a * np.sin(phi) * np.cos(theta)
    Y = b * np.sin(phi) * np.sin(theta)
    Z = c * np.cos(phi)
    return X, Y, Z

n_theta = 80
n_phi   = 50
theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
phi   = np.linspace(0, np.pi, n_phi)

Theta, Phi = np.meshgrid(theta, phi)
Xg, Yg, Zg = ellipsoid_xyz(Theta, Phi)

theta_min, theta_max = 1.2, 2.1
phi_min,   phi_max   = 0.7, 1.25

mask = (Theta >= theta_min) & (Theta <= theta_max) & (Phi >= phi_min) & (Phi <= phi_max)

Theta_loc = Theta[mask].ravel()
Phi_loc   = Phi[mask].ravel()

X_loc = Xg[mask].ravel()
Y_loc = Yg[mask].ravel()
Z_loc = Zg[mask].ravel()

# Points for Delaunay in parameter space:
P2 = np.column_stack((Theta_loc, Phi_loc))

tri = Delaunay(P2)
faces = tri.simplices.copy()  # (ntri, 3)

def circumcircle_2d(A, B, C, eps=1e-12):
    ax, ay = A
    bx, by = B
    cx, cy = C

    d = 2 * (ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < eps:
        return None  # degenerate triangle

    ax2ay2 = ax*ax + ay*ay
    bx2by2 = bx*bx + by*by
    cx2cy2 = cx*cx + cy*cy

    ux = (ax2ay2*(by-cy) + bx2by2*(cy-ay) + cx2cy2*(ay-by)) / d
    uy = (ax2ay2*(cx-bx) + bx2by2*(ax-cx) + cx2cy2*(bx-ax)) / d

    center = np.array([ux, uy])
    r = np.linalg.norm(center - A)
    return center, r

def points_inside_circle(center, r, points, tri_idx, tol=1e-10):
    verts = set(tri_idx.tolist())
    d = np.linalg.norm(points - center, axis=1)
    inside = (d < (r - tol))
    for vid in verts:
        inside[vid] = False
    return np.where(inside)[0]  


K = 8
if len(faces) <= K:
    chosen_ids = np.arange(len(faces))
else:
    chosen_ids = np.linspace(0, len(faces)-1, K, dtype=int)

chosen_faces = faces[chosen_ids]

circles = []
inside_lists = []
for t in chosen_faces:
    A, B, C = P2[t[0]], P2[t[1]], P2[t[2]]
    cc = circumcircle_2d(A, B, C)
    if cc is None:
        circles.append(None)
        inside_lists.append(None)
        continue
    center, r = cc
    circles.append((center, r))
    inside_lists.append(points_inside_circle(center, r, P2, t, tol=1e-10))

def add_tri_edges_2d(fig, P2, faces_in, color="rgba(0,0,0,0.45)", width=1.5):
    xs, ys = [], []
    for a_i, b_i, c_i in faces_in:
        ring = [a_i, b_i, c_i, a_i]
        for u, v in zip(ring[:-1], ring[1:]):
            xs += [P2[u,0], P2[v,0], None]
            ys += [P2[u,1], P2[v,1], None]
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="lines",
        line=dict(color=color, width=width),
        showlegend=False
    ))

def add_wireframe_3d(fig, Xp, Yp, Zp, faces_in, color="rgba(0,0,0,0.9)", width=3):
    edges = set()
    for a_i, b_i, c_i in faces_in:
        edges.add(tuple(sorted((a_i, b_i))))
        edges.add(tuple(sorted((b_i, c_i))))
        edges.add(tuple(sorted((c_i, a_i))))
    xs, ys, zs = [], [], []
    for u, v in edges:
        xs += [Xp[u], Xp[v], None]
        ys += [Yp[u], Yp[v], None]
        zs += [Zp[u], Zp[v], None]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(color=color, width=width),
        showlegend=False
    ))

def setup2d(fig, title):
    fig.update_layout(
        title=title,
        width=980, height=720,
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis=dict(title="θ (theta)", gridcolor="lightgray", zeroline=False),
        yaxis=dict(title="φ (phi)", gridcolor="lightgray", zeroline=False),
    )

def setup3d(fig, title):
    fig.update_layout(
        title=title,
        width=980, height=760,
        paper_bgcolor="white",
        scene=dict(
            bgcolor="white",
            aspectmode="data",
            camera=dict(eye=dict(x=2.0, y=1.9, z=1.6)),
            xaxis=dict(gridcolor="lightgray", showbackground=False),
            yaxis=dict(gridcolor="lightgray", showbackground=False),
            zaxis=dict(gridcolor="lightgray", showbackground=False),
        )
    )


figA = go.Figure()
figA.add_trace(go.Scatter(
    x=P2[:,0], y=P2[:,1],
    mode="markers",
    marker=dict(size=7, color="royalblue", opacity=0.9),
    showlegend=False
))
setup2d(figA, "Քայլ 1 Լոկալ կետերը ")
figA.show()

figB = go.Figure()
figB.add_trace(go.Scatter(
    x=P2[:,0], y=P2[:,1],
    mode="markers",
    marker=dict(size=6, color="royalblue", opacity=0.85),
    showlegend=False
))
add_tri_edges_2d(figB, P2, faces, color="rgba(0,0,0,0.35)", width=1.2)
setup2d(figB, "Քայլ 2 Դելոնեյի եռանկյունացում ")
figB.show()


figC = go.Figure()


add_tri_edges_2d(figC, P2, faces, color="rgba(0,0,0,0.18)", width=1.0)


figC.add_trace(go.Scatter(
    x=P2[:,0], y=P2[:,1],
    mode="markers",
    marker=dict(size=6, color="midnightblue", opacity=0.75),
    showlegend=False
))

for idx, t in enumerate(chosen_faces):
   
    add_tri_edges_2d(figC, P2, np.array([t]), color="rgba(255,0,0,0.95)", width=3.2)

    cc = circles[idx]
    if cc is None:
        continue
    center, r = cc

    ang = np.linspace(0, 2*np.pi, 250)
    cx = center[0] + r*np.cos(ang)
    cy = center[1] + r*np.sin(ang)

    # circumcircle
    figC.add_trace(go.Scatter(
        x=cx, y=cy,
        mode="lines",
        line=dict(color="rgba(0,160,0,0.85)", width=2.4),
        showlegend=False
    ))

    # circle center label: in = how many points lie inside the circle (should be 0)
    inside_idx = inside_lists[idx]
    in_count = 0 if inside_idx is None else len(inside_idx)

    figC.add_trace(go.Scatter(
        x=[center[0]], y=[center[1]],
        mode="markers+text",
        marker=dict(size=10, color="black"),
        text=[f"in={in_count}"],
        textposition="top center",
        showlegend=False
    ))

  
    if in_count > 0:
        figC.add_trace(go.Scatter(
            x=P2[inside_idx,0], y=P2[inside_idx,1],
            mode="markers",
            marker=dict(size=10, color="orange", opacity=0.95),
            showlegend=False
        ))

setup2d(figC, "Քայլ 3 ")
figC.show()


figD = go.Figure()

figD.add_trace(go.Mesh3d(
    x=X_loc, y=Y_loc, z=Z_loc,
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    intensity=Z_loc,               # nice color shading by height
    colorscale="Viridis",
    showscale=True,
    flatshading=True,
    opacity=0.92,
    showlegend=False
))
add_wireframe_3d(figD, X_loc, Y_loc, Z_loc, faces, color="rgba(0,0,0,0.95)", width=2)

# points (vertices)
figD.add_trace(go.Scatter3d(
    x=X_loc, y=Y_loc, z=Z_loc,
    mode="markers",
    marker=dict(size=3, color="gold", opacity=0.95),
    showlegend=False
))

setup3d(figD, "Քայլ 4 եռանկյունացում ")
figD.show()


figE = go.Figure()

figE.add_trace(go.Mesh3d(
    x=X_loc, y=Y_loc, z=Z_loc,
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    intensity=Z_loc,
    colorscale="Viridis",
    showscale=False,
    flatshading=True,
    opacity=0.65,
    showlegend=False
))

add_wireframe_3d(figE, X_loc, Y_loc, Z_loc, faces, color="rgba(0,0,0,0.25)", width=1)

# chosen triangles strong edges
add_wireframe_3d(figE, X_loc, Y_loc, Z_loc, chosen_faces, color="rgba(255,0,0,0.98)", width=6)

setup3d(figE, "Քայլ 5 Ընտրված եռանկյուններ (կարմիր) 3D մակերևույթի վրա")
figE.show()
